# 02. Modeling & Evaluation Pipeline
## Customer Sentiment Analysis & Reporting for E-commerce

This notebook walks through the machine learning pipeline:
1. Custom text preprocessing (tokenization, stop words preserving negations, lemmatization)
2. Feature representation comparison: Bag-of-Words vs. TF-IDF with a Naive Bayes classifier
3. Inspecting the active model selected by `make train` (baselines, Logistic Regression, Random Forest and Gradient Boosting candidates, Grid/Randomized search with stratified CV; full comparison in `reports/evaluation.md`)
4. Final evaluation on held-out test split (macro-F1 >= 0.65)
5. Ethical bias evaluation across departments and age groups

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

from src.data.clean import build_model_input_text
from src.models.train import load_dataset_from_db_or_clean
from src.features.text import TextPreprocessor, build_bow_vectorizer, build_tfidf_vectorizer
from src.models.predict import ModelService
from src.models.evaluate import compute_metrics, run_ethics_evaluation

### 1. Load Stratified Splits
Ensuring strict isolation: train (70%), validation (15%), test (15%).

In [ ]:
df = load_dataset_from_db_or_clean()
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

print(f'Train split: {len(train_df)} rows')
print(f'Validation split: {len(val_df)} rows')
print(f'Test split: {len(test_df)} rows')

### 1b. Feature Engineering: Bag-of-Words vs. TF-IDF
Text is cleaned once with `TextPreprocessor`, then each representation feeds a Multinomial Naive Bayes classifier scored on the validation split.

In [ ]:
X_train, y_train = build_model_input_text(train_df), train_df['true_label']
X_val, y_val = build_model_input_text(val_df), val_df['true_label']

cleaner = TextPreprocessor()
X_train_clean, X_val_clean = cleaner.transform(X_train), cleaner.transform(X_val)
print('Example cleaned text:', X_train_clean[0][:120])

for name, vectorizer in [('BoW', build_bow_vectorizer()), ('TF-IDF', build_tfidf_vectorizer())]:
    nb_pipe = Pipeline([('vectorizer', vectorizer), ('classifier', MultinomialNB())])
    nb_pipe.fit(X_train_clean, y_train)
    m = compute_metrics(y_val, nb_pipe.predict(X_val_clean))
    print(f'{name:7s} + MultinomialNB -> val macro-F1 {m["macro_f1"]:.4f}, accuracy {m["accuracy"]:.4f}')

### 2. Inspect Active Model Pipeline & Artifacts

In [ ]:
service = ModelService()
print(f'Active model version: {service.version}')
print('Pipeline structure:', service.pipeline)

### 3. Final Test Set Evaluation
Evaluated on held-out test set once with complete metrics and confusion matrix.

In [ ]:
# The model was trained on Title + Review Text, so score the same input
y_test_pred = service.pipeline.predict(build_model_input_text(test_df))
metrics = compute_metrics(test_df['true_label'], y_test_pred)

print(f'Test Accuracy:  {metrics["accuracy"]:.4f}')
print(f'Test Macro-F1:  {metrics["macro_f1"]:.4f} (Requirement >= 0.65)')
print(f'Test Weighted-F1: {metrics["weighted_f1"]:.4f}')
print('\nPer-class metrics:')
for label, scores in metrics['per_class'].items():
    print(f'  {label:8s} -> Precision: {scores["precision"]:.4f}, Recall: {scores["recall"]:.4f}, F1: {scores["f1"]:.4f}')

### 4. Ethical Bias & Demographic Evaluation
Inspecting performance across product departments and customer age bands (<30, 30-45, >45).

In [ ]:
ethics_results = run_ethics_evaluation(test_df, test_df['true_label'].tolist(), list(y_test_pred))
print(f'Overall Macro-F1: {ethics_results["overall_macro_f1"]:.4f}')
print('Department Breakdown:')
for dept, res in ethics_results['department_metrics'].items():
    print(f'  {dept:15s} (n={res["count"]:4d}) -> Macro-F1: {res["macro_f1"]:.4f} ({res["diff_from_overall"]:+.4f})')

print('\nAge Group Breakdown:')
for group, res in ethics_results['age_band_metrics'].items():
    print(f'  {group:15s} (n={res["count"]:4d}) -> Macro-F1: {res["macro_f1"]:.4f} ({res["diff_from_overall"]:+.4f})')

print('\nFlagged disparities (> 10 points):', ethics_results['disparities'] or 'none')

### 5. Interactive Prediction Verification

In [ ]:
sample_texts = [
    'This dress fits like a glove and the silk fabric is breathtaking!',
    'It is okay, but runs slightly small and the color is duller than pictured.',
    'Terrible quality! Shrunk in cold wash and seams ripped on day one. Do not buy.'
]
results = service.predict_batch(sample_texts, persist=False)['predictions']
for text, res in zip(sample_texts, results):
    print(f'Review: "{text}"')
    print(f'  -> Predicted: {res["label"]} (confidence: {res["confidence"]:.3f})')
    print(f'  -> Probabilities: {res["probabilities"]}\n')